[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ofermend/hands-on-rag/blob/main/chapter3/split-pdf.ipynb)

# Chapter 3: PDF Splitting for Large Documents

When working with large PDF documents in RAG systems, it's often necessary to split them into smaller chunks for processing. This is useful because:

1. **Memory constraints**: Large PDFs may exceed memory limits during parsing
2. **Parallel processing**: Smaller chunks can be processed concurrently
3. **Incremental updates**: Only re-process changed sections

This notebook demonstrates how to split a PDF into smaller files based on page count using PyPDF2.

**What you'll learn:**
- Download and open PDFs from URLs for processing
- Split large PDFs into smaller page-based chunks
- Write individual chunks as separate PDF files

**Prerequisites:** `pip install PyPDF2 requests`

In [1]:
!pip install --quiet PyPDF2


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
import requests
import io
import math
from urllib.parse import urlparse
from PyPDF2 import PdfReader, PdfWriter

## PDF Splitting Functions

The `split_pdf` function downloads a PDF from a URL and splits it into smaller files:
- Downloads the PDF into memory
- Calculates the number of chunks based on `pages_per_chunk`
- Creates separate PDF files for each chunk

In [3]:
def get_pdf_reader(input_source):
    """
    Gets a PdfReader object from a URL.

    Args:
        input_source (str): The URL of the PDF.

    Returns:
        tuple: (PdfReader object, base filename, total pages)
    """
    base_filename = "output" # Default base filename
    response = requests.get(input_source, stream=True, timeout=30)
    response.raise_for_status()  # Raise an exception for bad status codes (4xx or 5xx)

    # Get filename from URL path
    parsed_url = urlparse(input_source)
    path_part = os.path.basename(parsed_url.path)
    if path_part and '.' in path_part: # Basic check for filename extension
         base_filename = os.path.splitext(path_part)[0]

    # Read content into memory
    pdf_content = io.BytesIO(response.content)
    reader = PdfReader(pdf_content)
    total_pages = len(reader.pages)
    print(f"Successfully downloaded and opened PDF from URL. Total pages: {total_pages}")
    return reader, base_filename, total_pages

def split_pdf(input_source, output_dir, pages_per_chunk):
    """
    Splits a PDF file (from local path or URL) into multiple smaller PDF files.

    Args:
        input_source (str): The path to the local PDF file or the URL of the PDF.
        output_dir (str): The directory where the smaller PDF chunks will be saved.
        pages_per_chunk (int): The maximum number of pages each chunk should have.
    """
    reader, base_filename, total_pages = get_pdf_reader(input_source)

    if reader is None:
        print("Failed to get PDF reader. Aborting split.")
        return

    try:
        # Create the output directory if it doesn't exist
        os.makedirs(output_dir, exist_ok=True)
        print(f"Output directory '{output_dir}' ensured.")

        # Calculate the number of chunks
        num_chunks = math.ceil(total_pages / pages_per_chunk)       
        print(f"Splitting into {num_chunks} chunks of max {pages_per_chunk} pages each.")

        # Process each chunk
        for i in range(num_chunks):
            writer = PdfWriter()
            start_page = i * pages_per_chunk
            # Ensure end_page doesn't exceed total_pages
            end_page = min(start_page + pages_per_chunk, total_pages)

            print(f"Processing chunk {i+1}/{num_chunks} (pages {start_page + 1}-{end_page})...")

            # Add pages to the new PDF chunk
            for page_num in range(start_page, end_page):
                writer.add_page(reader.pages[page_num])

            # Construct the output filename using the determined base filename
            output_filename = os.path.join(output_dir, f"{base_filename}_chunk_{i+1}.pdf")

            # Write the chunk to a new PDF file
            with open(output_filename, 'wb') as outfile:
                writer.write(outfile)
            print(f"Chunk {i+1} saved as '{output_filename}'")

        print("\nPDF splitting completed successfully!")

    except Exception as e:
        print(f"An error occurred during the splitting process: {e}")

Let's split the Sutton & Barto Reinforcement Learning textbook (352 pages) into 50-page chunks.

In [4]:
pdf_link = "https://web.stanford.edu/class/psych209/Readings/SuttonBartoIPRLBook2ndEd.pdf"
output_folder = "pdf_chunks"                   # Name of the folder to save the chunks
pages_per_split = 50                           # Number of pages per output PDF file
split_pdf(pdf_link, output_folder, pages_per_split)


Successfully downloaded and opened PDF from URL. Total pages: 352
Output directory 'pdf_chunks' ensured.
Splitting into 8 chunks of max 50 pages each.
Processing chunk 1/8 (pages 1-50)...
Chunk 1 saved as 'pdf_chunks/SuttonBartoIPRLBook2ndEd_chunk_1.pdf'
Processing chunk 2/8 (pages 51-100)...
Chunk 2 saved as 'pdf_chunks/SuttonBartoIPRLBook2ndEd_chunk_2.pdf'
Processing chunk 3/8 (pages 101-150)...
Chunk 3 saved as 'pdf_chunks/SuttonBartoIPRLBook2ndEd_chunk_3.pdf'
Processing chunk 4/8 (pages 151-200)...
Chunk 4 saved as 'pdf_chunks/SuttonBartoIPRLBook2ndEd_chunk_4.pdf'
Processing chunk 5/8 (pages 201-250)...
Chunk 5 saved as 'pdf_chunks/SuttonBartoIPRLBook2ndEd_chunk_5.pdf'
Processing chunk 6/8 (pages 251-300)...
Chunk 6 saved as 'pdf_chunks/SuttonBartoIPRLBook2ndEd_chunk_6.pdf'
Processing chunk 7/8 (pages 301-350)...
Chunk 7 saved as 'pdf_chunks/SuttonBartoIPRLBook2ndEd_chunk_7.pdf'
Processing chunk 8/8 (pages 351-352)...
Chunk 8 saved as 'pdf_chunks/SuttonBartoIPRLBook2ndEd_chunk_8.pd